# Data Validation — Precipitation Nowcasting

補充 eda.ipynb 沒有的驗證項目：
1. H×W 尺寸確認（各衛星 + GPM 是否一致）
2. 空間對齊確認（CRS / bounds / pixel size）
3. 夜間 VIS 波段檢查（0 or NaN？）
4. 波段物理波長對照表（跨衛星 band mapping）

In [1]:
from pathlib import Path
import ast
import numpy as np
import pandas as pd
import rasterio

DATA_DIR  = Path("/Volumes/T7/new_code/solafune/data")
CSV_TRAIN = DATA_DIR / "train_dataset.csv"
SATELLITE_SUBDIR = {'himawari': 'himawari', 'goes': 'goes', 'meteosat': 'meteosat'}

df = pd.read_csv(CSV_TRAIN)
print(f"Loaded {len(df)} rows")

Loaded 40686 rows


## 1. H×W 尺寸確認

每顆衛星的影像大小 + 對應 GPM 影像是否相同。
這決定了模型輸入 shape，也確認是否需要 resize。

In [2]:
print(f"{'Satellite':<12} {'Sat H×W':<14} {'GPM H×W':<14} {'Match?'}")
print('-' * 50)

for sat in ['himawari', 'goes', 'meteosat']:
    # 找第一筆本地有檔案的樣本
    for _, row in df[df['satellite_target'] == sat].iterrows():
        fnames = ast.literal_eval(row['last_30_minutes_observation_filename'])
        if not fnames:
            continue
        sat_path = DATA_DIR / SATELLITE_SUBDIR[sat] / fnames[0]
        gpm_path = DATA_DIR / 'gpm_imerg' / row['gpm_imerg_filename']
        if not sat_path.exists() or not gpm_path.exists():
            continue

        with rasterio.open(sat_path) as s:
            sat_h, sat_w = s.height, s.width
            sat_bands = s.count
        with rasterio.open(gpm_path) as g:
            gpm_h, gpm_w = g.height, g.width

        match = '✅' if (sat_h == gpm_h and sat_w == gpm_w) else '❌ MISMATCH'
        print(f"{sat:<12} {sat_h}×{sat_w} ({sat_bands}ch)   {gpm_h}×{gpm_w}          {match}")
        break

Satellite    Sat H×W        GPM H×W        Match?
--------------------------------------------------
himawari     81×81 (16ch)   41×41          ❌ MISMATCH
goes         141×141 (16ch)   41×41          ❌ MISMATCH
meteosat     144×144 (16ch)   41×41          ❌ MISMATCH


## 2. 空間對齊確認（CRS / Bounds / Pixel Size）

確認衛星影像和 GPM 標籤的空間範圍是否對齊。
若 bounds 不同代表兩者覆蓋範圍不同，需要 reproject/crop。

In [3]:
for sat in ['himawari', 'goes', 'meteosat']:
    for _, row in df[df['satellite_target'] == sat].iterrows():
        fnames = ast.literal_eval(row['last_30_minutes_observation_filename'])
        if not fnames:
            continue
        sat_path = DATA_DIR / SATELLITE_SUBDIR[sat] / fnames[0]
        gpm_path = DATA_DIR / 'gpm_imerg' / row['gpm_imerg_filename']
        if not sat_path.exists() or not gpm_path.exists():
            continue

        with rasterio.open(sat_path) as s:
            sat_crs    = s.crs
            sat_bounds = s.bounds
            sat_res    = s.res
        with rasterio.open(gpm_path) as g:
            gpm_crs    = g.crs
            gpm_bounds = g.bounds
            gpm_res    = g.res

        print(f"=== {sat} ===")
        print(f"  Satellite CRS    : {sat_crs}")
        print(f"  GPM       CRS    : {gpm_crs}")
        print(f"  CRS match        : {'✅' if sat_crs == gpm_crs else '❌'}")
        print(f"  Satellite bounds : {sat_bounds}")
        print(f"  GPM       bounds : {gpm_bounds}")
        print(f"  Bounds match     : {'✅' if sat_bounds == gpm_bounds else '❌ DIFFERENT'}")
        print(f"  Satellite res    : {sat_res}")
        print(f"  GPM       res    : {gpm_res}")
        print()
        break

=== himawari ===
  Satellite CRS    : None
  GPM       CRS    : None
  CRS match        : ✅
  Satellite bounds : BoundingBox(left=0.0, bottom=81.0, right=81.0, top=0.0)
  GPM       bounds : BoundingBox(left=0.0, bottom=41.0, right=41.0, top=0.0)
  Bounds match     : ❌ DIFFERENT
  Satellite res    : (1.0, 1.0)
  GPM       res    : (1.0, 1.0)

=== goes ===
  Satellite CRS    : None
  GPM       CRS    : None
  CRS match        : ✅
  Satellite bounds : BoundingBox(left=0.0, bottom=141.0, right=141.0, top=0.0)
  GPM       bounds : BoundingBox(left=0.0, bottom=41.0, right=41.0, top=0.0)
  Bounds match     : ❌ DIFFERENT
  Satellite res    : (1.0, 1.0)
  GPM       res    : (1.0, 1.0)

=== meteosat ===
  Satellite CRS    : None
  GPM       CRS    : None
  CRS match        : ✅
  Satellite bounds : BoundingBox(left=0.0, bottom=144.0, right=144.0, top=0.0)
  GPM       bounds : BoundingBox(left=0.0, bottom=41.0, right=41.0, top=0.0)
  Bounds match     : ❌ DIFFERENT
  Satellite res    : (1.0, 1.0)
 

## 3. 夜間 VIS 波段檢查

VIS（可見光）波段在夜間無太陽光，數值會是 0 或 NaN。
這影響標準化策略：若是 0 則 z-score 後變成負值，需要 mask 或特別處理。

In [4]:
# VIS 波段 index（0-based）
VIS_BANDS = {
    'himawari': [0, 1, 2, 3, 4, 5],    # B01–B06
    'goes':     [0, 1, 2, 3, 4, 5],    # C01–C06
    'meteosat': [0, 1, 2, 3, 4],       # vis_04–vis_09
}

# 找夜間樣本（00:00–05:00 UTC 通常為夜間）
df['hour'] = pd.to_datetime(df['datetime']).dt.hour
night_df = df[df['hour'].between(0, 4)]

print(f"{'Satellite':<12} {'VIS Band':<10} {'Min':>8} {'Max':>8} {'Zero%':>8} {'NaN%':>8}")
print('-' * 58)

for sat in ['himawari', 'goes', 'meteosat']:
    found = False
    for _, row in night_df[night_df['satellite_target'] == sat].iterrows():
        fnames = ast.literal_eval(row['last_30_minutes_observation_filename'])
        if not fnames:
            continue
        sat_path = DATA_DIR / SATELLITE_SUBDIR[sat] / fnames[0]
        if not sat_path.exists():
            continue
        with rasterio.open(sat_path) as src:
            img = src.read().astype(np.float32)  # (16, H, W)

        for b_idx in VIS_BANDS[sat]:
            band = img[b_idx].ravel()
            nan_pct  = np.isnan(band).mean() * 100
            zero_pct = (band == 0).mean() * 100
            print(f"{sat:<12} B{b_idx+1:<9} {band[~np.isnan(band)].min() if nan_pct < 100 else float('nan'):>8.2f} "
                  f"{band[~np.isnan(band)].max() if nan_pct < 100 else float('nan'):>8.2f} "
                  f"{zero_pct:>7.1f}% {nan_pct:>7.1f}%")
        found = True
        break
    if not found:
        print(f"{sat:<12} (no local night sample found)")

Satellite    VIS Band        Min      Max    Zero%     NaN%
----------------------------------------------------------
himawari     B1             1.00    14.00     0.0%     0.0%
himawari     B2             1.00    12.00     0.0%     0.0%
himawari     B3             1.00     8.00     0.0%     0.0%
himawari     B4             1.00    10.00     0.0%     0.0%
himawari     B5             0.00     7.00    22.3%     0.0%
himawari     B6             0.00     5.00    29.3%     0.0%
goes         B1            14.00   112.00     0.0%     0.0%
goes         B2             6.00   106.00     0.0%     0.0%
goes         B3             3.00   123.00     0.0%     0.0%
goes         B4             0.00    94.00     0.6%     0.0%
goes         B5             0.00    76.00     0.0%     0.0%
goes         B6             0.00    81.00     0.7%     0.0%
meteosat     B1             0.00     2.00    77.5%     0.0%
meteosat     B2             0.00     2.00    68.7%     0.0%
meteosat     B3             0.00     2.00

## 4. 波段物理波長對照表（Band Mapping）

三顆衛星的波段名稱不同，但物理波長接近。
這個表確認跨衛星的「對應通道」，方便之後做特徵工程。

In [5]:
# idx: (band_name, center_wavelength_um, type)
BAND_MAP = {
    'himawari': [
        ('B01', 0.47,  'VIS'), ('B02', 0.51,  'VIS'), ('B03', 0.64,  'VIS'),
        ('B04', 0.86,  'NIR'), ('B05', 1.6,   'SWIR'),('B06', 2.3,   'SWIR'),
        ('B07', 3.9,   'IR'),  ('B08', 6.2,   'WV'),  ('B09', 6.9,   'WV'),
        ('B10', 7.3,   'WV'),  ('B11', 8.6,   'IR'),  ('B12', 9.6,   'O3'),
        ('B13', 10.4,  'IR'),  ('B14', 11.2,  'IR'),  ('B15', 12.4,  'IR'),
        ('B16', 13.3,  'IR'),
    ],
    'goes': [
        ('C01', 0.47,  'VIS'), ('C02', 0.64,  'VIS'), ('C03', 0.86,  'NIR'),
        ('C04', 1.38,  'SWIR'),('C05', 1.6,   'SWIR'),('C06', 2.2,   'SWIR'),
        ('C07', 3.9,   'IR'),  ('C08', 6.2,   'WV'),  ('C09', 6.9,   'WV'),
        ('C10', 7.3,   'WV'),  ('C11', 8.5,   'IR'),  ('C12', 9.6,   'O3'),
        ('C13', 10.3,  'IR'),  ('C14', 11.2,  'IR'),  ('C15', 12.3,  'IR'),
        ('C16', 13.3,  'IR'),
    ],
    'meteosat': [
        ('vis_04', 0.444,'VIS'),('vis_05', 0.506,'VIS'),('vis_06', 0.640,'VIS'),
        ('vis_08', 0.865,'NIR'),('vis_09', 0.914,'NIR'),('nir_13', 1.380,'SWIR'),
        ('nir_16', 1.610,'SWIR'),('nir_22',2.250,'SWIR'),('ir_38', 3.800,'IR'),
        ('wv_63',  6.300,'WV'), ('wv_73',  7.350,'WV'), ('ir_87', 8.700,'IR'),
        ('ir_97',  9.660,'O3'), ('ir_105',10.500,'IR'), ('ir_123',12.300,'IR'),
        ('ir_133',13.300,'IR'),
    ],
}

print(f"{'Idx':<5} {'Himawari':<10} {'µm':<6} {'GOES':<10} {'µm':<6} {'Meteosat':<10} {'µm':<6} {'Type'}")
print('-' * 70)
for i in range(16):
    h = BAND_MAP['himawari'][i]
    g = BAND_MAP['goes'][i]
    m = BAND_MAP['meteosat'][i]
    print(f"{i+1:<5} {h[0]:<10} {h[1]:<6} {g[0]:<10} {g[1]:<6} {m[0]:<10} {m[1]:<6} {h[2]}")

Idx   Himawari   µm     GOES       µm     Meteosat   µm     Type
----------------------------------------------------------------------
1     B01        0.47   C01        0.47   vis_04     0.444  VIS
2     B02        0.51   C02        0.64   vis_05     0.506  VIS
3     B03        0.64   C03        0.86   vis_06     0.64   VIS
4     B04        0.86   C04        1.38   vis_08     0.865  NIR
5     B05        1.6    C05        1.6    vis_09     0.914  SWIR
6     B06        2.3    C06        2.2    nir_13     1.38   SWIR
7     B07        3.9    C07        3.9    nir_16     1.61   IR
8     B08        6.2    C08        6.2    nir_22     2.25   WV
9     B09        6.9    C09        6.9    ir_38      3.8    WV
10    B10        7.3    C10        7.3    wv_63      6.3    WV
11    B11        8.6    C11        8.5    wv_73      7.35   IR
12    B12        9.6    C12        9.6    ir_87      8.7    O3
13    B13        10.4   C13        10.3   ir_97      9.66   IR
14    B14        11.2   C14        11

In [6]:
# 關鍵通道摘要
print("=== 雲頂溫度通道（降水預測最重要）===")
for sat, bands in BAND_MAP.items():
    for i, (name, wl, typ) in enumerate(bands):
        if typ == 'IR' and 10.0 <= wl <= 11.0:
            print(f"  {sat:<12} idx={i}  {name}  {wl}µm")

print()
print("=== 水氣通道 ===")
for sat, bands in BAND_MAP.items():
    for i, (name, wl, typ) in enumerate(bands):
        if typ == 'WV':
            print(f"  {sat:<12} idx={i}  {name}  {wl}µm")

=== 雲頂溫度通道（降水預測最重要）===
  himawari     idx=12  B13  10.4µm
  goes         idx=12  C13  10.3µm
  meteosat     idx=13  ir_105  10.5µm

=== 水氣通道 ===
  himawari     idx=7  B08  6.2µm
  himawari     idx=8  B09  6.9µm
  himawari     idx=9  B10  7.3µm
  goes         idx=7  C08  6.2µm
  goes         idx=8  C09  6.9µm
  goes         idx=9  C10  7.3µm
  meteosat     idx=9  wv_63  6.3µm
  meteosat     idx=10  wv_73  7.35µm


## 5. 驗證小結

### 驗證結果總結

### 1. H×W（⚠️ 重大發現：衛星與 GPM 尺寸不一致）

| 衛星 | 衛星影像 H×W | GPM H×W | 一致? |
|------|-------------|---------|-------|
| Himawari | **81×81** | 41×41 | ❌ |
| GOES | **141×141** | 41×41 | ❌ |
| Meteosat | **144×144** | 41×41 | ❌ |

**結論：** GPM 永遠是 41×41，衛星影像是 GPM 的 2–3.5x 大。  
→ `dataset.py` 需在 `__getitem__` 中將 GPM resize 到衛星解析度（或將衛星 downsample 到 41×41）。  
→ 目前推薦方案：GPM `resize` 到衛星解析度，讓 model 預測衛星尺寸後再 downsample 回 41×41 比對。

---

### 2. CRS / Bounds / Resolution

| 項目 | 衛星 | GPM | 一致? |
|------|------|-----|-------|
| CRS | None | None | ✅ |
| Bounds | 各不同（像素範圍） | (0,0,41,41) | ❌（不同像素空間） |
| Resolution | (1.0, 1.0) | (1.0, 1.0) | ✅ |

→ CRS = None = 無地理投影，資料在 preprocessing 時已做空間對齊。  
→ 不需要 reproject，但需要處理解析度差異。

---

### 3. 夜間 VIS 波段

- **NaN：無（所有波段 0%）** ✅
- 夜間 = 0（非 NaN）—— z-score 後為負值
- Himawari B1–B4（UTC 00:00–04:00）：min=1（亞洲白天仍有光）
- Himawari B5–B6（SWIR）：22–29% 為 0
- Meteosat VIS：64–82% 為 0（歐洲/非洲樣本為夜間）
- mask channel 已記錄每個 frame 有效性，model 可學會自動處理夜間

---

### 4. Band Mapping（關鍵通道索引）

| 衛星 | 雲頂溫度 (10µm) | index | 水氣通道 |
|------|----------------|-------|---------|
| Himawari | B13 (10.4µm) | **12** | idx 7, 8, 9 |
| GOES | C13 (10.3µm) | **12** | idx 7, 8, 9 |
| Meteosat | ir_105 (10.5µm) | **13** ⚠️ | idx 9, 10 |

**⚠️ 注意：** Meteosat 雲頂溫度 index 比另外兩顆多 1。若要統一提取特徵，需要按衛星名稱切換 index。

---

### 5. 對模型的影響

1. **架構**：UNet decoder 輸出尺寸需設計為衛星解析度，最後 crop/resize 到 41×41 計算 loss。  
   或：dataset 中將所有衛星和 GPM 都 resize 到統一解析度（建議 81×81 或 41×41）。
2. **正規化**：VIS 夜間為 0 是正常值，不需要特殊處理（z-score 會產生負值）。
3. **波段**：若使用雲頂溫度 band 作為特定特徵（attention/loss weighting），需依衛星名稱用不同 idx。